# 📦 BLOQUE 00: INSTALACIÓN DE LIBRERÍAS Y DEPENDENCIAS
---
> **Objetivo:** Instalar todos los paquetes necesarios de **LangChain**, **Google Gemini**, **ChromaDB** y procesadores de imagen dentro del entorno de Python de Jupyter.

In [15]:
!pip install -q langchain langchain-google-genai langchain-community langchain-chroma chromadb pillow pandas ipywidgets

print("Instalacion completa.")

Instalacion completa.


# 🛠️ BLOQUE 01: CONFIGURACIÓN E INICIALIZACIÓN DE GEMINI
---
> **Objetivo:** Cargar la clave de API de Google Gemini e inicializar el modelo de lenguaje y el sistema de embeddings directamente desde la sesión.

In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Se declara la clave de API en las variables de entorno
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6LrXzQthTzq0wV5NtYgnnJ0Hb-GU-6-h8Ak3SKlYL2nEA"

MODELO_LLM = "gemini-3.1-flash-lite"
MODELO_EMBEDDING = "gemini-embedding-2-preview"

# Se declara el modelo LLM
llm = ChatGoogleGenerativeAI(model=MODELO_LLM, temperature=0)

# Se declara el modelo de embeddings
embeddings = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDING)

# Se acciona el ping de verificacion
print(llm.invoke("Responde unicamente: 'Gemini conectado.' y nada mas.").content)

[{'type': 'text', 'text': 'Gemini conectado.', 'extras': {'signature': 'EjQKMgERTTIPI/2P3xMqVeT7d2TmIGZm7Rfmypw5r4Rez+HxGwOX9B5v4EAXnpuTsxIkmxNG'}}]


# 📚 BLOQUE 02: INGESTIÓN Y CREACIÓN DE VECTOR STORES (RAG)
---
> **Objetivo:** Leer la base documental de **Patito S.A.**, fragmentar los textos (*chunking*) y construir los tres almacenes vectoriales en memoria usando **Chroma**.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# Se declara la configuracion de archivos a procesar
docs_config = {
    "contratos": "01_Clausulas_Contractuales.txt",
    "datos": "02_Proteccion_Datos.txt",
    "cumplimiento": "03_Cumplimiento_Etica.txt"
}

vectorstores = {}
retrievers = {}

# Se declara el separador de texto en fragmentos
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Se acciona la lectura, fraccionamiento e indexacion en Chroma DB
for key, filename in docs_config.items():
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Archivo no encontrado: {filename}")
        
    loader = TextLoader(filename, encoding="utf-8")
    documents = loader.load()
    chunks = text_splitter.split_documents(documents)
    
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=f"patito_{key}"
    )
    
    vectorstores[key] = vectorstore
    retrievers[key] = vectorstore.as_retriever(search_kwargs={"k": 2})

C:\Users\Gael Mora\AppData\Local\Temp\ipykernel_8648\4119211739.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


# 📖 BLOQUE 03: AGENTES ESPECIALIZADOS DE LECTURA (RAG)
---
> **Objetivo:** Crear las herramientas de consulta para los tres agentes (*Contratos*, *Protección de Datos* y *Cumplimiento*), restringiendo sus respuestas estrictamente al contexto extraído.

In [3]:
from langchain.tools import tool

PROMPT_RAG_BASE = """Eres un especialista legal del departamento Legal de Patito S.A.
Responde a la consulta utilizando ÚNICAMENTE la información del contexto proporcionado.

REGLA STRICTA: Si el contexto no contiene suficiente información para responder con certeza, debes responder exactamente:
"No encontré información suficiente en la base documental proporcionada."

No inventes ni agregues conocimiento externo al contexto entregado.

Contexto recuperado:
{context}

Consulta:
{query}
"""

@tool
def consultar_contratos(query: str) -> str:
    """Consulta la base de conocimiento sobre cláusulas estándar, tipos de contrato, plazos y proceso de firma."""
    docs = retrievers["contratos"].invoke(query)
    if not docs:
        return "No encontré información suficiente en la base documental proporcionada."
    context = "\n---\n".join([d.page_content for d in docs])
    prompt = PROMPT_RAG_BASE.format(context=context, query=query)
    res = llm.invoke(prompt)
    return res.content

@tool
def consultar_datos_personales(query: str) -> str:
    """Consulta la base de conocimiento sobre tratamiento de datos, retención, brechas y derechos."""
    docs = retrievers["datos"].invoke(query)
    if not docs:
        return "No encontré información suficiente en la base documental proporcionada."
    context = "\n---\n".join([d.page_content for d in docs])
    prompt = PROMPT_RAG_BASE.format(context=context, query=query)
    res = llm.invoke(prompt)
    return res.content

@tool
def consultar_cumplimiento(query: str) -> str:
    """Consulta la base de conocimiento sobre código de ética, regalos, sobornos y conflictos de interés."""
    docs = retrievers["cumplimiento"].invoke(query)
    if not docs:
        return "No encontré información suficiente en la base documental proporcionada."
    context = "\n---\n".join([d.page_content for d in docs])
    prompt = PROMPT_RAG_BASE.format(context=context, query=query)
    res = llm.invoke(prompt)
    return res.content

print("✅ Agentes de lectura (RAG) definidos y encapsulados como herramientas.")

✅ Agentes de lectura (RAG) definidos y encapsulados como herramientas.


# ⚡ BLOQUE 04: AGENTE DE ACCIÓN (REGISTRO DE SOLICITUDES EN TEXTO)
---
> **Objetivo:** Implementar la herramienta de escritura con validación de campos obligatorios, generación de ID único y persistencia en el archivo `registro_solicitudes_legal.txt`.

In [4]:
import uuid
from datetime import datetime

@tool
def registrar_solicitud_contrato(
    tipo_contrato: str,
    datos_proveedor: str,
    objeto_servicio: str,
    plazo: str,
    monto: str,
    trata_datos_personales: str,
    confirmado: bool = False
) -> str:
    """Registra una solicitud de revisión de contrato en un archivo de texto local tras validar datos requeridos."""
    
    faltantes = []
    if not tipo_contrato or tipo_contrato == "None": faltantes.append("Tipo de Contrato")
    if not datos_proveedor or datos_proveedor == "None": faltantes.append("Datos del Proveedor")
    if not objeto_servicio or objeto_servicio == "None": faltantes.append("Objeto/Servicio")
    if not plazo or plazo == "None": faltantes.append("Plazo")
    if not monto or monto == "None": faltantes.append("Monto")
    if not trata_datos_personales or trata_datos_personales == "None": faltantes.append("Tratamiento de Datos Personales")
    
    if faltantes:
        return f"Error: No se puede registrar la solicitud. Faltan los siguientes campos obligatorios: {', '.join(faltantes)}."
        
    if not confirmado:
        return (
            "VALIDACIÓN CORRECTA. Todos los datos obligatorios están presentes.\n"
            "Resumen del registro a crear:\n"
            f"- Tipo de Contrato: {tipo_contrato}\n"
            f"- Proveedor: {datos_proveedor}\n"
            f"- Objeto: {objeto_servicio}\n"
            f"- Plazo: {plazo}\n"
            f"- Monto: {monto}\n"
            f"- Trata Datos Personales: {trata_datos_personales}\n\n"
            "Por favor, confirme la operación indicando 'confirmado=True' para escribir en el archivo."
        )

    os.makedirs("./registros", exist_ok=True)
    archivo_registro = "./registros/registro_solicitudes_legal.txt"
    
    id_solicitud = f"PAT-{uuid.uuid4().hex[:8].upper()}"
    fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    linea_registro = (
        f"ID: {id_solicitud} | FECHA: {fecha_hora} | TIPO: {tipo_contrato} | "
        f"PROVEEDOR: {datos_proveedor} | OBJETO: {objeto_servicio} | "
        f"PLAZO: {plazo} | MONTO: {monto} | DATOS_PERS: {trata_datos_personales}\n"
    )
    
    with open(archivo_registro, "a", encoding="utf-8") as f:
        f.write(linea_registro)
        
    return f"REGISTRO EXITOSO: La solicitud fue guardada con el ID {id_solicitud} el {fecha_hora}."

print("✅ Agente de Acción con validaciones configurado.")

✅ Agente de Acción con validaciones configurado.


# 👁️ BLOQUE 05: AGENTE MULTIMODAL DE IMAGEN (VISIÓN GEMINI)
---
> **Objetivo:** Integrar la capacidad visual de Gemini para procesar imágenes de contratos o documentos escaneados y validar su contenido.

In [5]:
import PIL.Image

@tool
def analizar_imagen_contrato(ruta_imagen: str, pregunta: str) -> str:
    """Analiza una imagen o escaneo de contrato usando la capacidad de visión de Gemini para extraer texto o validar elementos."""
    if not os.path.exists(ruta_imagen):
        return f"Error: La ruta de la imagen no existe: {ruta_imagen}"
        
    try:
        imagen = PIL.Image.open(ruta_imagen)
        
        prompt_multimodal = (
            "Eres un auditor legal de Patito S.A. "
            f"Analiza la imagen adjunta y responde a la siguiente pregunta: {pregunta}"
        )
        
        res = llm.invoke([prompt_multimodal, imagen])
        return res.content
    except Exception as e:
        return f"Error al procesar la imagen: {str(e)}"

print("✅ Agente Multimodal listo para inspeccionar imágenes.")

✅ Agente Multimodal listo para inspeccionar imágenes.


# 🧠 BLOQUE 06: AGENTE ORQUESTADOR PRINCIPAL
---
> **Objetivo:** Definir la lógica de enrutamiento y delegación central que coordina las herramientas de los subagentes y sintetiza la respuesta final.

In [6]:
HERRAMIENTAS_MESA_AYUDA = [
    consultar_contratos,
    consultar_datos_personales,
    consultar_cumplimiento,
    registrar_solicitud_contrato,
    analizar_imagen_contrato
]

PROMPT_ORQUESTADOR = """Eres el Agente Orquestador Central de la Mesa de Ayuda IA del Departamento Legal de Patito S.A.

Analiza la consulta del usuario, identifica qué áreas intervienen y coordina la ejecución de las herramientas requeridas.

Herramientas disponibles:
- `consultar_contratos`: Cláusulas, tipos de contrato, plazos y firmas.
- `consultar_datos_personales`: Protección de datos, conservación de registros, derechos o brechas.
- `consultar_cumplimiento`: Regalos, código de ética, conflictos de interés o corrupción.
- `registrar_solicitud_contrato`: Solicitudes explícitas de registro de contratos.
- `analizar_imagen_contrato`: Consultas que adjunten imágenes de documentos.

Instrucciones:
1. Llama a todas las herramientas que correspondan a la pregunta.
2. Combina los resultados en un reporte integrado.
3. Lista explícitamente los agentes que intervinieron y los documentos/fuentes consultados.

Consulta:
{input}
"""

def orquestar_consulta(consulta_usuario: str, ruta_imagen: str = None) -> str:
    """Procesa la pregunta del usuario, enruta la solicitud y entrega la respuesta final consolidada."""
    llm_con_herramientas = llm.bind_tools(HERRAMIENTAS_MESA_AYUDA)
    
    prompt = PROMPT_ORQUESTADOR.format(input=consulta_usuario)
    if ruta_imagen:
        prompt += f"\nNOTA: Se ha adjuntado una imagen en la ruta: {ruta_imagen}"
        
    respuesta_inicial = llm_con_herramientas.invoke(prompt)
    
    if not respuesta_inicial.tool_calls:
        return respuesta_inicial.content
        
    resultados_herramientas = []
    
    for tool_call in respuesta_inicial.tool_calls:
        nombre_herramienta = tool_call["name"]
        argumentos = tool_call["args"]
        
        for tool_obj in HERRAMIENTAS_MESA_AYUDA:
            if tool_obj.name == nombre_herramienta:
                res_tool = tool_obj.invoke(argumentos)
                resultados_herramientas.append({
                    "agente": nombre_herramienta,
                    "resultado": res_tool
                })
                break
                
    prompt_sintesis = f"""Sintetiza una respuesta final clara y estructurada para el usuario usando los resultados devueltos por los agentes:

Resultados obtenidos:
{resultados_herramientas}

Consulta original:
{consulta_usuario}

Asegúrate de detallar los agentes intervinientes y el soporte documental utilizado.
"""
    
    respuesta_final = llm.invoke(prompt_sintesis)
    return respuesta_final.content

print("✅ Agente Orquestador configurado correctamente.")

✅ Agente Orquestador configurado correctamente.


# 🧪 BLOQUE 07: INTERFAZ DE CHATBOT INTERACTIVO Y CONSERSACIONAL CON MEMORIA
---
> **Objetivo:** iniciaruna sesión de chat continua que recupere contexto de la base de datos vectorial en tiempo real y mantenga el historial de la conversación.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Plantilla de prompt con memoria e instrucciones
prompt_chat = ChatPromptTemplate.from_messages([
    ("system", "Responde la pregunta del usuario basándote únicamente en el siguiente contexto extraído de los documentos. Si la información no está en el contexto, indica amablemente que no la encuentras.\n\nContexto:\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

historial = []

# Función para consultar la base vectorial activa
def responder_chat(pregunta, modulo="contratos"):
    global historial
    
    docs = retrievers[modulo].invoke(pregunta)
    contexto = "\n\n".join([doc.page_content for doc in docs])
    
    chain = prompt_chat | llm
    respuesta = chain.invoke({
        "context": contexto,
        "chat_history": historial,
        "question": pregunta
    })
    
    historial.append(HumanMessage(content=pregunta))
    historial.append(AIMessage(content=respuesta.content))
    
    return respuesta.content

# Bucle interactivo de Chatbot
print("=" * 50)
print("🤖 Chatbot RAG activo al final del script.")
print("Escribe 'salir' para terminar la conversación.")
print("=" * 50)

while True:
    usuario = input("\nTú: ")
    if usuario.lower() in ["salir", "exit", "quit"]:
        print("🤖 Chat finalizado.")
        break
        
    if not usuario.strip():
        continue
        
    respuesta = responder_chat(usuario, modulo="contratos")
    print(f"\nIA: {respuesta}")

🤖 Chatbot RAG activo al final del script.
Escribe 'salir' para terminar la conversación.



Tú:  ¿Cuáles son las causas de terminación o rescisión del contrato?



IA: [{'type': 'text', 'text': 'De acuerdo con el contexto proporcionado, el contrato debe incluir las **causales de terminación y sus consecuencias**. Sin embargo, el documento no especifica cuáles son esas causas en particular, solo indica que deben estar presentes como una cláusula mínima.', 'extras': {'signature': 'EjQKMgERTTIPOfb28ddwFxJorM/IcVxc9QL/pL0nsyP2jA3bg5eN17Num339B6uu73pxjmJU'}}]
